# VAANI Dynamic Price Assistant

Runnable end-to-end prototype for dynamic price recommendation.

### Flow
`Product attributes → SerpAPI / mock market data → cleaning & feature extraction → XGBoost demo model → profitable competitive price range`

### Requirements
```bash
pip install requests numpy pandas scikit-learn xgboost
```

For live SerpAPI mode, set the `SERPAPI_KEY` environment variable before running the market-data cell.


## 1. Imports

In [1]:
from __future__ import annotations

import os
import re
from dataclasses import dataclass

import numpy as np
import pandas as pd
import requests
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor


## 2. Product Data Model

In [2]:
# class Product:
#     title: str
#     description: str
#     category: str
#     material: str
#     material_cost: float
#     labour_cost: float
#     packaging_cost: float
#     other_cost: float
#     handmade: int = 1

#     @property
#     def query(self) -> str:
#         return f"{self.title} {self.material} {self.category} handmade India"

#     @property
#     def total_cost(self) -> float:
#         return self.material_cost + self.labour_cost + self.packaging_cost + self.other_cost

In [3]:
@dataclass
class Product:
    title: str
    description: str
    category: str
    material: str
    material_cost: float
    labour_cost: float
    packaging_cost: float
    other_cost: float
    handmade: int = 1

    @property
    def query(self) -> str:
        return f"{self.title} {self.material} {self.category} handmade India"

    @property
    def total_cost(self) -> float:
        return self.material_cost + self.labour_cost + self.packaging_cost + self.other_cost


MOCK_MARKET_RESULTS = [
    {"title": "Handmade Sheesham Wood Jewellery Box", "price": "₹999", "source": "Craft Store"},
    {"title": "Wooden Hand Painted Jewelry Box", "price": "₹1,199", "source": "Artisan Mart"},
    {"title": "Sheesham Jewellery Storage Box Handmade", "price": "₹1,099", "source": "Decor Shop"},
    {"title": "Handcrafted Wooden Jewellery Box", "price": "₹899", "source": "India Crafts"},
    {"title": "Premium Handmade Wooden Jewelry Box", "price": "₹1,349", "source": "Home Bazaar"},
    {"title": "Handmade Sheesham Wood Jewellery Box", "price": "₹999", "source": "Craft Store"},
    {"title": "Plastic Jewellery Organizer", "price": "₹299", "source": "Irrelevant Shop"},
    {"title": "Antique Luxury Jewellery Cabinet", "price": "₹9,999", "source": "Outlier Shop"},
]

## 3. Demo Market Data

In [4]:
# MOCK_MARKET_RESULTS = [
#     {"title": "Handmade Sheesham Wood Jewellery Box", "price": "₹999", "source": "Craft Store"},
#     {"title": "Wooden Hand Painted Jewelry Box", "price": "₹1,199", "source": "Artisan Mart"},
#     {"title": "Sheesham Jewellery Storage Box Handmade", "price": "₹1,099", "source": "Decor Shop"},
#     {"title": "Handcrafted Wooden Jewellery Box", "price": "₹899", "source": "India Crafts"},
#     {"title": "Premium Handmade Wooden Jewelry Box", "price": "₹1,349", "source": "Home Bazaar"},
#     {"title": "Handmade Sheesham Wood Jewellery Box", "price": "₹999", "source": "Craft Store"},
#     {"title": "Plastic Jewellery Organizer", "price": "₹299", "source": "Irrelevant Shop"},
#     {"title": "Antique Luxury Jewellery Cabinet", "price": "₹9,999", "source": "Outlier Shop"},
# ]

## 4. Fetch Market Results

In [5]:
def fetch_market_results(query: str) -> tuple[list[dict], str]:
    """Fetch Google Shopping results when a key exists; otherwise use demo data."""
    api_key = os.getenv("SERPAPI_KEY", "591711fca815d031ba964782f864af035c557deae3f49a0a4148caafa78b1a65").strip()
    if not api_key:
        return MOCK_MARKET_RESULTS, "DEMO data (no API key supplied)"

    response = requests.get(
        "https://serpapi.com/search.json",
        params={"engine": "google_shopping", "q": query, "gl": "in", "hl": "en", "api_key": api_key},
        timeout=30,
    )
    response.raise_for_status()
    payload = response.json()
    rows = payload.get("shopping_results", [])
    if not rows:
        raise RuntimeError("API returned no shopping_results. Check query/key/quota.")
    return rows, "LIVE SerpAPI Google Shopping data"

## 5. Price Parsing

In [6]:
def parse_price(value) -> float | None:
    """Convert ₹1,099 / 1099.0 / extracted_price into a number."""
    if isinstance(value, (int, float)):
        return float(value)
    match = re.search(r"\d[\d,]*(?:\.\d+)?", str(value or ""))
    return float(match.group(0).replace(",", "")) if match else None

## 6. Market Data Cleaning

In [7]:
def clean_market_data(rows: list[dict], product: Product) -> pd.DataFrame:
    """Normalize, remove duplicates/irrelevant rows, then IQR-filter outliers."""
    required_words = {product.material.lower(), product.category.lower().split()[0]}
    cleaned = []
    for row in rows:
        title = str(row.get("title", "")).strip()
        price = parse_price(row.get("extracted_price", row.get("price")))
        title_lower = title.lower()
        if not title or price is None or price <= 0:
            continue
        if not any(word in title_lower for word in required_words):
            continue
        cleaned.append({"title": title, "price": price, "seller": row.get("source", "Unknown")})

    frame = pd.DataFrame(cleaned).drop_duplicates(subset=["title", "price"])
    if len(frame) < 3:
        raise RuntimeError("Too few relevant products after cleaning; broaden the search query.")
    q1, q3 = frame["price"].quantile([0.25, 0.75])
    iqr = q3 - q1
    frame = frame[frame["price"].between(q1 - 1.5 * iqr, q3 + 1.5 * iqr)]
    return frame.reset_index(drop=True)


## 7. Feature Engineering

In [8]:

def market_features(frame: pd.DataFrame, product: Product) -> dict[str, float]:
    prices = frame["price"]
    return {
        "total_cost": product.total_cost,
        "market_min": float(prices.min()),
        "market_mean": float(prices.mean()),
        "market_median": float(prices.median()),
        "market_max": float(prices.max()),
        "similar_count": float(len(prices)),
        "handmade": float(product.handmade),
    }


## 8. Train XGBoost Prototype Model

In [9]:
def train_demo_xgboost(seed: int = 42) -> tuple[XGBRegressor, float]:
    """Train on synthetic prototype data. Replace this with real historical sales CSV later."""
    rng = np.random.default_rng(seed)
    n = 3000
    cost = rng.uniform(150, 2500, n)
    median = cost * rng.uniform(1.25, 2.4, n)
    market_min = median * rng.uniform(0.70, 0.94, n)
    market_max = median * rng.uniform(1.08, 1.55, n)
    mean = median * rng.uniform(0.96, 1.08, n)
    count = rng.integers(3, 80, n)
    handmade = rng.integers(0, 2, n)
    # This simulates the price at which comparable products successfully sold.
    sold_price = np.maximum(
        cost * 1.18,
        0.58 * median + 0.20 * mean + 0.10 * market_min + 0.05 * market_max
        + handmade * 0.07 * median + rng.normal(0, 35, n),
    )
    X = pd.DataFrame({
        "total_cost": cost, "market_min": market_min, "market_mean": mean,
        "market_median": median, "market_max": market_max,
        "similar_count": count, "handmade": handmade,
    })
    X_train, X_test, y_train, y_test = train_test_split(X, sold_price, test_size=0.2, random_state=seed)
    model = XGBRegressor(
        n_estimators=250, max_depth=4, learning_rate=0.05,
        subsample=0.85, colsample_bytree=0.9, objective="reg:squarederror",
        random_state=seed, n_jobs=4,
    )
    model.fit(X_train, y_train)
    return model, float(model.score(X_test, y_test))

## 9. Retail Price Rounding

In [10]:
def round_price(value: float) -> int:
    """Round to familiar Indian retail endings: nearest hundred minus one."""
    return max(99, int(round((value + 1) / 100) * 100 - 1))

## 10. Price Recommendation Logic

In [11]:
def recommend(product: Product, features: dict[str, float], model: XGBRegressor) -> dict:
    predicted = float(model.predict(pd.DataFrame([features]))[0])
    minimum_profitable = product.total_cost * 1.18  # demo: 18% minimum margin on cost
    suggested = min(max(predicted, minimum_profitable), features["market_max"] * 1.05)
    low = max(minimum_profitable, suggested * 0.92, features["market_min"])
    high = min(suggested * 1.10, features["market_max"])
    suggested = min(max(suggested, low), high)
    return {
        "minimum_profitable": round_price(minimum_profitable),
        "recommended_low": round_price(low),
        "suggested_price": round_price(suggested),
        "recommended_high": round_price(high),
        "estimated_margin_percent": round((suggested - product.total_cost) / suggested * 100, 1),
    }

## 11. Run the Full Prototype

In [12]:
def main() -> None:
    # In the full app, Qwen2.5-VL/LLM will produce these structured fields.
    product = Product(
        title="Hand-painted Sheesham wood jewellery box",
        description="Medium handmade jewellery box made by an artisan.",
        category="jewellery box",
        material="wood",
        material_cost=320,
        labour_cost=250,
        packaging_cost=50,
        other_cost=80,
    )

    raw_rows, data_source = fetch_market_results(product.query)
    cleaned = clean_market_data(raw_rows, product)
    features = market_features(cleaned, product)
    model, demo_r2 = train_demo_xgboost()
    result = recommend(product, features, model)

    print("\nVAANI — DYNAMIC PRICE ASSISTANT")
    print(f"Data source              : {data_source}")
    print(f"Search query             : {product.query}")
    print(f"Raw → cleaned products   : {len(raw_rows)} → {len(cleaned)}")
    print(f"Clean market prices      : {cleaned['price'].astype(int).tolist()}")
    print(f"Production cost          : ₹{product.total_cost:,.0f}")
    print(f"Market range             : ₹{features['market_min']:,.0f} – ₹{features['market_max']:,.0f}")
    print(f"Market median            : ₹{features['market_median']:,.0f}")
    print(f"Prototype model R²       : {demo_r2:.3f} (synthetic data; not production accuracy)")
    print("-" * 54)
    print(f"Minimum profitable price : ₹{result['minimum_profitable']:,}")
    print(f"Recommended range        : ₹{result['recommended_low']:,} – ₹{result['recommended_high']:,}")
    print(f"Suggested price          : ₹{result['suggested_price']:,}")
    print(f"Estimated margin         : {result['estimated_margin_percent']}%")


if __name__ == "__main__":
    main()



VAANI — DYNAMIC PRICE ASSISTANT
Data source              : LIVE SerpAPI Google Shopping data
Search query             : Hand-painted Sheesham wood jewellery box wood jewellery box handmade India
Raw → cleaned products   : 40 → 37
Clean market prices      : [69, 248, 499, 745, 262, 279, 398, 299, 360, 599, 199, 1009, 424, 483, 899, 895, 498, 499, 699, 399, 249, 651, 599, 519, 849, 1299, 658, 390, 499, 345, 267, 298, 430, 499, 899, 398, 799]
Production cost          : ₹700
Market range             : ₹69 – ₹1,299
Market median            : ₹498
Prototype model R²       : 0.999 (synthetic data; not production accuracy)
------------------------------------------------------
Minimum profitable price : ₹799
Recommended range        : ₹799 – ₹899
Suggested price          : ₹799
Estimated margin         : 15.3%
